# Deep Learning from Scratch: NumPy MLP and Adam Solver

This notebook demonstrates the core matrix calculus and optimization routines behind deep neural networks by building a 2-layer Multi-Layer Perceptron (MLP) with **Sigmoid and ReLU activations**, a **Binary Cross-Entropy Loss** objective, and a **from-scratch Adam Optimizer** using only raw NumPy.

## 1. Mathematical Formulas & Vectorization

We use Andrew Ng's vectorized dimension standards where each training sample is stored in a single column:
- $X = A^{[0]} \in \mathbb{R}^{n_0 \times m}$ (where $m$ is sample size, $n_0$ is input feature count).

### Layer 1 (Hidden, ReLU):
$$Z^{[1]} = W^{[1]} A^{[0]} + b^{[1]}$$
$$A^{[1]} = \max(0, Z^{[1]})$$

### Layer 2 (Output, Sigmoid):
$$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$$
$$A^{[2]} = \sigma(Z^{[2]}) = \frac{1}{1 + e^{-Z^{[2]}}}$$

### Loss Function (Binary Cross-Entropy):
$$L = -\frac{1}{m} \sum_{i=1}^m \left[ y_i \ln(a_i^{[2]}) + (1 - y_i) \ln(1 - a_i^{[2]}) \right]$$

## 2. Generate Synthetic Classification Dataset

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed
np.random.seed(42)

# Generate concentric circle classification data (non-linear)
n_samples = 500
r = np.random.uniform(0, 1.2, n_samples)
theta = np.random.uniform(0, 2 * np.pi, n_samples)

X_raw = np.zeros((n_samples, 2))
X_raw[:, 0] = r * np.cos(theta)
X_raw[:, 1] = r * np.sin(theta)

# Target: Class 1 is inner ring, Class 0 is outer ring
y_raw = (r < 0.65).astype(int)

# Add noise
X_raw += np.random.normal(0, 0.05, X_raw.shape)

# Train/test split using pure NumPy random permutations
indices = np.random.permutation(n_samples)
split_idx = int(0.7 * n_samples)
train_idx, test_idx = indices[:split_idx], indices[split_idx:]

X_train_raw, X_test_raw = X_raw[train_idx], X_raw[test_idx]
y_train_raw, y_test_raw = y_raw[train_idx], y_raw[test_idx]

# Transpose arrays to fit column-vector standard (features x samples)
X_train = X_train_raw.T
y_train = y_train_raw.reshape(1, -1)
X_test = X_test_raw.T
y_test = y_test_raw.reshape(1, -1)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")

# Plot the data
plt.figure(figsize=(6, 4.5))
plt.scatter(X_raw[:, 0], X_raw[:, 1], c=y_raw, cmap='coolwarm', edgecolors='k', s=25)
plt.title("Non-Linear Concentric Rings classification")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

## 3. Implement activations, forward prop, and loss functions

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def d_relu(z):
    return np.where(z > 0, 1.0, 0.0)

def compute_loss(A2, Y):
    m = Y.shape[1]
    # Small epsilon prevents log(0) numeric overflows
    eps = 1e-15
    loss = -1.0 / m * np.sum(Y * np.log(A2 + eps) + (1.0 - Y) * np.log(1.0 - A2 + eps))
    return loss

## 4. Implement Backpropagation Math

The derivatives derived via the Chain Rule are:
- $dZ^{[2]} = A^{[2]} - Y$
- $dW^{[2]} = \frac{1}{m} dZ^{[2]} (A^{[1]})^T$
- $db^{[2]} = \frac{1}{m} \sum dZ^{[2]}$ (row sum, keeping shape `(n_2, 1)`)
- $dZ^{[1]} = (W^{[2]})^T dZ^{[2]} \odot g^{[1]\prime}(Z^{[1]})$
- $dW^{[1]} = \frac{1}{m} dZ^{[1]} (A^{[0]})^T$
- $db^{[1]} = \frac{1}{m} \sum dZ^{[1]}$

In [ ]:
def forward_prop(X, params):
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    
    cache = {'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}
    return A2, cache

def backward_prop(X, Y, cache, params):
    m = Y.shape[1]
    W2 = params['W2']
    A1, A2 = cache['A1'], cache['A2']
    Z1 = cache['Z1']
    
    # Backprop calculations
    dZ2 = A2 - Y
    dW2 = 1.0 / m * np.dot(dZ2, A1.T)
    db2 = 1.0 / m * np.sum(dZ2, axis=1, keepdims=True)
    
    dZ1 = np.dot(W2.T, dZ2) * d_relu(Z1)
    dW1 = 1.0 / m * np.dot(dZ1, X.T)
    db1 = 1.0 / m * np.sum(dZ1, axis=1, keepdims=True)
    
    grads = {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}
    return grads

## 5. Implement Adam Optimizer from Scratch

Adam updates weights using moving averages of gradients ($v$) and squared gradients ($s$):
$$v = \beta_1 v + (1 - \beta_1) dW$$
$$s = \beta_2 s + (1 - \beta_2) dW^2$$
$$v^{\text{corrected}} = \frac{v}{1 - \beta_1^t}, \quad s^{\text{corrected}} = \frac{s}{1 - \beta_2^t}$$
$$W \leftarrow W - \frac{\alpha}{\sqrt{s^{\text{corrected}}} + \epsilon} v^{\text{corrected}}$$

In [ ]:
class AdamOptimizer:
    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        
        # Initialize moments to zeros
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.s = {k: np.zeros_like(v) for k, v in params.items()}
        
    def update(self, params, grads):
        self.t += 1
        for key in params.keys():
            grad = grads['d' + key]
            
            # Compute first and second moment averages
            self.v[key] = self.beta1 * self.v[key] + (1 - self.beta1) * grad
            self.s[key] = self.beta2 * self.s[key] + (1 - self.beta2) * np.square(grad)
            
            # Correct bias
            v_corr = self.v[key] / (1.0 - self.beta1 ** self.t)
            s_corr = self.s[key] / (1.0 - self.beta2 ** self.t)
            
            # Update weights
            params[key] -= self.lr * v_corr / (np.sqrt(s_corr) + self.eps)
        return params

## 6. Initialize Weights (He Initialization) & Run Training Loop

In [ ]:
# Network architecture dimensions
n_0 = X_train.shape[0]  # 2 inputs
n_1 = 8                 # 8 hidden neurons
n_2 = 1                 # 1 output neuron

# He (Kaiming) Initialization
params = {
    'W1': np.random.randn(n_1, n_0) * np.sqrt(2.0 / n_0),
    'b1': np.zeros((n_1, 1)),
    'W2': np.random.randn(n_2, n_1) * np.sqrt(2.0 / n_1),
    'b2': np.zeros((n_2, 1))
}

# Instantiate optimizer
optimizer = AdamOptimizer(params, lr=0.01)

# Training loop
epochs = 200
loss_history = []

for epoch in range(epochs + 1):
    # 1. Forward Pass
    A2, cache = forward_prop(X_train, params)
    
    # 2. Compute Loss
    loss = compute_loss(A2, y_train)
    loss_history.append(loss)
    
    # 3. Backward Pass
    grads = backward_prop(X_train, y_train, cache, params)
    
    # 4. Update Weights via Adam
    params = optimizer.update(params, grads)
    
    if epoch % 20 == 0:
        # Validate test performance
        A2_test, _ = forward_prop(X_test, params)
        test_loss = compute_loss(A2_test, y_test)
        # Calculate accuracy
        test_acc = np.mean((A2_test > 0.5) == y_test)
        print(f"Epoch {epoch:03d} | Train Loss: {loss:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}")

## 7. Visualize Convergence

Let's check the loss trajectory curve.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.plot(loss_history, color='#339af0', linewidth=2.5, label='BCE Log-Loss')
plt.title("NumPy MLP Training Loss Convergence with Adam")
plt.xlabel("Epoch")
plt.ylabel("Log-Loss")
plt.legend()
plt.show()